In [1]:
from dotenv import load_dotenv
load_dotenv('/Users/adityanarvekar/agentic/agents/.env')

True

In [2]:
from langchain_openrouter import ChatOpenRouter




In [3]:
# Initialize ChatOpenRouter with a free model
llm = ChatOpenRouter(
    model="openrouter/owl-alpha",
    temperature=0.7
)



In [18]:
llm_nvidia = ChatOpenRouter(model="nvidia/nemotron-3-super-120b-a12b:free", temperature=0)


In [4]:
# Invoke the model
response = llm.invoke("What is the distance between the Earth and the Moon?")
print(response.content)

The average distance between the Earth and the Moon is about 384,400 kilometers (238,855 miles). This distance can vary slightly due to the Moon's elliptical orbit, ranging from about 363,300 km (225,700 miles) at its closest point (perigee) to about 405,500 km (251,900 miles) at its farthest point (apogee).


In [6]:
from pydantic import BaseModel, Field

class Answer(BaseModel):
    sentiment: str = Field(description="The sentiment of the review.")
    reasoning: str = Field(description="The reasoning for the sentiment analysis.")

In [15]:
llm_answer = llm.with_structured_output(Answer)
llm_nvidia_answer = llm_nvidia.with_structured_output(Answer)

In [8]:
response = llm_answer.invoke("What is the sentiment of the review: 'I love this product!'?")
print(response)

NotFoundResponseError: No endpoints found that support the provided 'tool_choice' value. To learn more about provider routing, visit: https://openrouter.ai/docs/guides/routing/provider-selection

In [17]:
response = llm_nvidia_answer.invoke("What is the sentiment of the review: 'I love this product!'?")
print(response)

sentiment='positive' reasoning="The review contains the word 'love' which expresses strong positive emotion toward the product."


In [ ]:
llm_structured = llm.with_structured_output(Answer, method="json_schema")

response = llm_structured.invoke(
    "Analyze this review's sentiment. "
    "Respond in JSON with keys: sentiment, reasoning.\n\n"
    "Review: 'The product was terrible.'"
)
print (response)

ResponseValidationError: Response validation failed: 6 validation errors for Unmarshaller
body.choices
  Field required [type=missing, input_value={'error': {'message': 'Pr...ed error', 'code': 502}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
body.created
  Field required [type=missing, input_value={'error': {'message': 'Pr...ed error', 'code': 502}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
body.id
  Field required [type=missing, input_value={'error': {'message': 'Pr...ed error', 'code': 502}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
body.model
  Field required [type=missing, input_value={'error': {'message': 'Pr...ed error', 'code': 502}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
body.object
  Field required [type=missing, input_value={'error': {'message': 'Pr...ed error', 'c

In [19]:
llm_nvidia_answer_json = llm_nvidia.with_structured_output(Answer, method="json_schema")

response = llm_nvidia_answer_json.invoke(
    "Analyze this review's sentiment. "
    "Respond in JSON with keys: sentiment, reasoning.\n\n"
    "Review: 'The product was terrible.'"
)
print (response)

sentiment=':negative' reasoning="The review contains the word 'terrible', which is a strong negative descriptor, indicating dissatisfaction with the product."


In [ ]:
from trustcall import create_extractor


extractor = create_extractor(llm_nvidia, tools=[Answer], tool_choice="Answer")

result = extractor.invoke({
    "messages": [("user", "Review: 'The product was absolutely terrible, would not recommend.'")]
})

answer = result["responses"][0]
print(answer.sentiment)
print(answer.reasoning)


negative
The review contains strong negative language such as 'absolutely terrible' and explicitly states they would not recommend the product, indicating dissatisfaction.


In [14]:
result

{'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to output sentiment and reasoning. The review: "The product was absolutely terrible, would not recommend." This is negative sentiment. Provide reasoning: words "absolutely terrible", "would not recommend". Use the Answer tool.\n\n', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to output sentiment and reasoning. The review: "The product was absolutely terrible, would not recommend." This is negative sentiment. Provide reasoning: words "absolutely terrible", "would not recommend". Use the Answer tool.\n\n'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'id': 'gen-1780563883-KqOwIS7CLEZ2DgL7mvlM', 'created': 1780563883, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019e91e0-b365-7841-a603-c08d1e785c6d-0', tool_calls=[{'name': 'Answer', 